In [ ]:
# from functools import reduce
# import pandas as pd
# from pathlib import Path
# from typing import List

# def merge_dfs(path_list: List[Path | str], 
#               on_cols: List[str] | str,
#               drop_cols: List[str] | str | None = None, 
#               how: str = "inner") -> pd.DataFrame:
    
#     if isinstance(on_cols, str):
#         on_cols = [on_cols]

#     if not on_cols:
#         raise ValueError("on_cols must be specified for merge")
    
#     dfs = [] 
#     cols = []
#     for path in path_list:
#         df = pd.read_csv(path, index_col=0)
        
#         if drop_cols:
#             df = df.drop(columns=drop_cols, errors="ignore") #.columns = [col for col in df.columns if col not in drop_cols]
        
#         for col in on_cols:
#             if col not in df.columns:
#                 raise ValueError(f"Column '{col}' is not in df '{path}'".)
            
#         cols.extend(df.columns)
#         dfs.append(df)
         
#     df_merged = reduce(
#                     lambda left, right: pd.merge(
#                         left,
#                         right,
#                         on=on_cols, 
#                         how=how
#                     ),
#                     dfs
#                 )
    
#     assert df_merged["case_id"].is_unique
#     assert not df_merged.isna().all(axis=1).any()

#     return df_merged
#     # pd.merge(df_list)

In [ ]:
# load df
# import pandas as pd

import utils.file_helper as fh
import core.logger as log

# check
import sys
sys.path[:3]

# 
logger = log.create_logger("notebook", "notebook")

folder = "/home/robfra/0_Portfolio_Projekte/LLM/data/data_processed/reports"
# /home/robfra/0_Portfolio_Projekte/LLM/data/data_processed/reports//home/robfra/0_Portfolio_Projekte/LLM/data/data_processed/reports/
file_3 = "2026-01-23_15:21:44_llm_v3_df.csv"
file_4 = "2026-01-22_19:58:24_llm_v4_df.csv"
file_1 = "2026-01-07_14:04:29_reports_llm_v1.csv"
file_2 = "2026-01-21_13:02:07_reports_llm_v1 (batch_size=5).csv"
# /home/robfra/0_Portfolio_Projekte/LLM/data/data_processed/reports/2026-01-23_14:09:53_llm_v3_df.csv
path_list = []
for file in [file_1, file_2, file_3, file_4]:
    path_comp = f"{folder}/{file}"
    path_list.append(path_comp)

drop_cols = ["report_text", "comment", "Unnamed: 0"]
on_cols = ["case_id", "domain", "clarity"]

suffix_col = "expected_action"
df_dict = fh.load_dfs(path_list, index_col=0)
# fh.df_preview(df_dict, logger)




Variables from .env.session loaded
Variables from .env loaded


In [3]:
import ast
from pathlib import Path
import os 


def col_name_correct(df_input, split_value):
    df = df_input.copy()

    col_to_rename = {}
    # name = ""
    for col in df.columns:
        parts = col.split(f"{split_value}")     # here: " "
        if len(parts) >= 2:
            col_to_rename[col] = parts[0]
            # name = parts[1]
        else:
            print(f"[INFO] No column name in df was splitted at '{split_value}'.")

    return df.rename(columns=col_to_rename)

def df_quick_check(df):
    print("[CHECK] SHAPE:\n", df.shape)
    print("\n[CHECK] HEAD:\n", df.head())
    return 

enlabel_dict = {True: "escalation",
                False: "no_escalation"}

def parse_list_str(x):
    if not isinstance(x, str) or x.strip() == "":
        return []
    try:
        return ast.literal_eval(x)
    except Exception:
        return []
    
def count_str_list_elements(x):
    x_new = parse_list_str(x)

    return len(x_new)


def extract_fn_fp(df, cols, enlabel_dict, split_value=None):
    if split_value:
        df = col_name_correct(df, " ") 

    df_lab = df.replace(enlabel_dict).copy()

    vals = list(enlabel_dict.values())

    df_fn = df_lab[(df_lab[cols[0]] == vals[0]) & 
                    (df_lab[cols[1]] == vals[1])]
    df_fp = df_lab[(df_lab[cols[0]] == vals[1]) & 
                    (df_lab[cols[1]] ==vals[0])]
        # df_3_rename["expected_action_llm"]]

    # for name, df in [("df_fn", df_fn), ("df_fp", df_fp)]: 
    #     print(f"[CHECK] df: {name.upper()}")
    #     df_quick_check(df)

    return {
        "all": df,
        "false_negative": df_fn,
        "false_postive": df_fp} 


def evaluate_fn_fp(df_dict, cols=None, folder=None):
    if cols is None:
        cols = ["n_risk_factors", "n_missing_information", "severity", 
                "confidence", "uncertainty", "confidence_derived"]

    if folder is None:
        folder = os.getenv("EVALUATED")
    
    # now = session.now
    
    for name, df in df_dict.items():
        df["n_risk_factors"] = df["risk_factors"].apply(count_str_list_elements)
        df["n_missing_information"] = df["missing_information"].apply(count_str_list_elements)

        f_path = Path(f"{folder}/{name}.csv")
        df.to_csv(f_path)
        # print(f"{name}:")
        # df_quick_check(df)  #.head())

        if name != "all":
            df_grouped = df.groupby("severity")[cols].value_counts()
            f_path_grouped = Path(f"{folder}/{name}_grouped.csv")
            df_grouped.to_csv(f_path_grouped)


In [4]:
import pandas as pd

df_3 = df_dict[f"{folder}/2026-01-23_15:21:44_llm_v3_df.csv"]

cols = ["expected_action", "expected_action_final"]
enlabel_dict = {True: "escalation",
                False: "no_escalation"}

dfs_to_compare = extract_fn_fp(df_3, cols, enlabel_dict) #, split_value=" ")

evaluate_fn_fp(dfs_to_compare)
# df_dict, cols=None, folder=None
    #     print(df_grouped)
#     num_col = []/home/robfra/0_Portfolio_Projekte/LLM/data
#     cat_col = []
#     for col in df:
#         if ??:

#         elif ???:

#         else:
#             print("WTF")

#     df[num_col].describe()
#     df[cat_col].describe()

#         df[cols].describe()
#     risk_factor_mean = np.mean(df["n_risk_factors"])
#     risk_factor_std = np.std(df["n_risk_factors"])

#     mis_info_mean = np.mean(df["n_missing_information"])
#     mis_info_std = np.std(df["n_missing_information"])

#     severe_mean = np.mean(df["severity"])
#     severe_std = np.std(df["n_risk_factors"])
# "case_id", 
#                 "severity", 'risk_factors', 'missing_information',
#                 "confidence", "uncertainty", "confidence_derived",

# print("original:\n", df_3.head())
# col_to_rename = {}
# for col in df_3.columns:
#     parts = col.split(" ")
#     if len(parts) >= 2:
#         col_to_rename[col] = parts[0]

# 
                
# df_3_rename = df_3.rename(columns=col_to_rename)    #.replace(enlabel_dict)
# # print("renamed:\n", df_3_rename.head())

# df_3_filt = df_3_rename[df_3_rename["expected_action"] != df_3_rename["expected_action_llm"]]

# cols_needed = ["case_id", "domain", "clarity", "severity", "confidence", "confidence_derived", "risk_factors", "missing_information"]

# print(pd.crosstab(
#     df_3_rename["expected_action"],
#     df_3_rename["expected_action_final"],
# ))


In [ ]:

false_negative:
[CHECK] SIZE:
 459

[CHECK] HEAD:
     Unnamed: 0  case_id   domain clarity expected_action  \
12          12       13  somatic   clear      escalation   
20          20       21  somatic   clear      escalation   
28          28       29  somatic   clear      escalation   
36          36       37  somatic   clear      escalation   
52          52       53  somatic   clear      escalation   

                                          report_text  \
12  The patient presents with acute somatic sympto...   
20  The patient presents with acute somatic sympto...   
28  The patient presents with acute somatic sympto...   
36  The patient presents with acute somatic sympto...   
52  The patient presents with acute somatic sympto...   

                                              comment  \
12  Potentially life-threatening somatic presentat...   
20  Potentially life-threatening somatic presentat...   
28  Potentially life-threatening somatic presentat...   
36  Potentially life-threatening somatic presentat...   
52  Potentially life-threatening somatic presentat...   

                                         risk_factors  \
12  ['acute somatic symptoms', 'chest discomfort',...   
20  ['rapidly worsening chest discomfort', 'shortn...   
28  ['acute somatic symptoms', 'chest discomfort',...   
36  ['acute somatic symptoms', 'chest discomfort',...   
52                                                 []   

                                  missing_information severity  \
12          ['specific diagnosis', 'medical history']     high   
20  ['specific diagnostic tests performed', "patie...     high   
28   ['specific diagnostic tests', 'medical history']     high   
36  ['specific diagnosis', 'exclusion of serious m...     high   
52                                                 []     high   

   expected_action_llm  confidence uncertainty  confidence_derived  \
12          escalation         0.3        high                 0.3   
20          escalation         0.3        high                 0.3   
28          escalation         0.4        high                 0.3   
36          escalation         0.3        high                 0.3   
52          escalation         0.3        high                 0.3   

   expected_action_final  n_risk_factors  n_missing_information  
12         no_escalation               3                      2  
20         no_escalation               3                      2  
28         no_escalation               3                      2  
36         no_escalation               3                      2  
52         no_escalation               0                      0  

######

false_postive:
[CHECK] SIZE:
 408

[CHECK] HEAD:
     Unnamed: 0  case_id domain clarity expected_action  \
9            9       10  psych   clear   no_escalation   
17          17       18  psych   clear   no_escalation   
33          33       34  psych   clear   no_escalation   
49          49       50  psych   clear   no_escalation   
57          57       58  psych   clear   no_escalation   

                                 risk_factors missing_information severity  \
9   ['emotional strain', 'ongoing stressors']                  []   medium   
17                                         []                  []   medium   
33  ['emotional strain', 'ongoing stressors']                  []   medium   
49  ['emotional strain', 'ongoing stressors']                  []   medium   
57                                         []                  []   medium   

   expected_action_llm  confidence uncertainty  confidence_derived  \
9           escalation         1.0         low                0.85   
17          escalation         1.0         low                0.85   
33          escalation         0.7         low                0.85   
49          escalation         1.0         low                0.85   
57          escalation         1.0         low                0.85   

   expected_action_final  n_risk_factors  n_missing_information  
9             escalation               2                      0  
17            escalation               0                      0  
33            escalation               2                      0  
49            escalation               2                      0  
57            escalation               0                      0  

In [2]:
# 
suffix_col = "expected_action"
df_merged = fh.merge_dfs(df_dict.values(), on_cols, suffix_col, drop_cols)

assert df_merged["case_id"].is_unique
assert not df_merged.isna().all(axis=1).any()


In [5]:
# clean df_merged
col_to_rename = {}
for col in df_merged.columns:
    parts = col.split(" ")
    if len(parts) >= 2:
        col_to_rename[col] = parts[0]

enlabel_dict = {True: "escalation",
                False: "no_escalation"}
                
df_renamed = df_merged.rename(columns=col_to_rename).replace(enlabel_dict)

df_merged_dict = {"df_merged": df_renamed,
                "df_v3": df_dict[f"{folder}/2026-01-23_14:08:04_llm_v3_df.csv"]}
print(df_renamed.head())
fh.df_preview(df_merged_dict, logger)

   case_id   domain    clarity expected_action_0 pred_llm_v1  \
0        1    psych      clear        escalation  escalation   
1        2    psych      clear     no_escalation  escalation   
2        3    psych  ambiguous        escalation  escalation   
3        4    psych  ambiguous     no_escalation  escalation   
4        5  somatic      clear        escalation  escalation   

   confidence_llm_v1 uncertainty_llm_v1  confidence_derived_llm_v1  \
0                0.8                low                       0.85   
1                0.9                low                       0.85   
2                0.7                low                       0.85   
3                0.9                low                       0.85   
4                0.4             medium                       0.60   

  expected_action_1 pred_llm_v1  ...  severity expected_action_llm  \
0        escalation  escalation  ...      high          escalation   
1     no_escalation  escalation  ...       low        

In [7]:
# save df
from datetime import datetime

now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

df_renamed.to_csv(f"/home/robfra/0_Portfolio_Projekte/LLM/data/data_processed/reports_merged/{now}_llm_1-4.csv")

In [10]:


col_pred = ["expected_action_2", 
            # "pred_llm_v1", 
            "expected_action_llm", 
            # "pred_llm_v4"
            ]
            # "expected_action_2", 
            # "expected_action_3", 
            # ]
df_pred = df_renamed.groupby(["domain", "clarity"])[col_pred]   # .copy() 

# print("\ndistribution predictions:")
print(df_pred.value_counts())

"""
distribution predictions:
domain   clarity    expected_action_0  pred_llm_v3  pred_llm_v4  
psych    ambiguous  escalation         escalation   escalation       39
                    no_escalation      escalation   escalation       38
                                                    no-escalation     2
                    escalation         escalation   no-escalation     1
         clear      escalation         escalation   escalation       40
                    no_escalation      escalation   escalation       38
                                                    no-escalation     2
somatic  ambiguous  escalation         escalation   escalation       40
                    no_escalation      escalation   escalation       40
         clear      escalation         escalation   escalation       40
                    no_escalation      escalation   escalation       40
Name: count, dtype: int64
"""

"""
run #1 --> LLM eskalation
run #1 --> LLM keine eskalation
run #1 --> LLM-eskalation mit AND-Variante deckeln
run #1 --> LLM-eskalation mit OR-Variante deckeln
"""

domain   clarity    expected_action_2  expected_action_llm
psych    ambiguous  escalation         escalation             39
                    no_escalation      escalation             39
                    escalation         no_escalation           1
                    no_escalation      no_escalation           1
         clear      escalation         escalation             40
                    no_escalation      escalation             39
                                       no_escalation           1
somatic  ambiguous  escalation         escalation             40
                    no_escalation      escalation             40
         clear      escalation         escalation             40
                    no_escalation      escalation             40
Name: count, dtype: int64


'\nrun #1 --> LLM eskalation\nrun #1 --> LLM keine eskalation\nrun #1 --> LLM-eskalation mit AND-Variante deckeln\nrun #1 --> LLM-eskalation mit OR-Variante deckeln\n'

In [8]:
cols_to_filt = ["case_id", 
                "severity", 'risk_factors', 'missing_information',
                "confidence", "uncertainty", "confidence_derived",
                # "confidence_llm_v4", "uncertainty_llm_v4", "confidence_derived_llm_v4"
                ]

df_filt = df_renamed[df_renamed["expected_action_0"] != df_renamed["expected_action_llm"]][cols_to_filt].copy()
            # (      (df_renamed["pred_llm_v1"].eq(True)) |
              # (.eq(True)))]
              # |
              # (df_renamed["pred_llm_v4"].eq(True)))][cols_to_filt].copy()
                    #  (df["pred_llm_v3"] == val) &
                    #  (df["pred_llm_v4"] == val)]
# print("false escalations")
# print(df_filt.head())

print("\ndescribe statistics")
print(df_filt) #["severity_llm_v3"].value_counts()) #.describe())
# def filter_cases(df, value_col, apply_col):
#     values = df[value_col].unique()
#     # if value_col not in apply_col:
#     #     apply_col.append(value_col)

#     for val in values:
#         cond = []
#         for i in len(apply_col):
#             cond.append([col] == val)
            
#         df_filt = df[(df["pred_llm_v1"] == val) &
#                      (df["pred_llm_v3"] == val) &
#                      (df["pred_llm_v4"] == val)]
#         print(f"{val}")


# for col in df_merged.columns():

# def merge_dfs(path_list: List[Path | str], 
#               on_cols: List[str] | str,
#               drop_cols: List[str] | str | None = None, 
#               how: str = "inner") -> pd.DataFrame:
# ['expected_action', 'confidence', 'uncertainty_level']
# ['case_id', 'domain', 'clarity', 'expected_action', 'report_text', 'comment']

# df_slim = df.drop(columns=["report_text", "comment", "case_id"])
# df_grouped = df_slim.groupby(["domain", "clarity"])[["expected_action", "pred_llm_v3 (batch_size=5)"]]   #.count()

# for cols in df_grouped.columns

# print(df_slim.head(5), "\n")
# print(df_grouped.value_counts())   # .head())




describe statistics
     case_id severity                               risk_factors  \
1          2      low                                         []   
3          4      low                                         []   
5          6      low                                         []   
7          8      low                                         []   
9         10   medium  ['emotional strain', 'ongoing stressors']   
..       ...      ...                                        ...   
311      312      low                                         []   
313      314   medium  ['emotional strain', 'ongoing stressors']   
315      316      low                                         []   
317      318      low                                         []   
319      320      low                                         []   

    missing_information  confidence uncertainty  confidence_derived  
1                    []        0.90         low                0.85  
3                    [

# Outputs
## baseline

---

## LLM run #1


--- 

## LLM run #2 (AND version)
```
domain   clarity    expected_action  pred_llm_v3
psych    ambiguous  escalation       True          40
                    no_escalation    True          40
         clear      escalation       True          40
                    no_escalation    True          40
somatic  ambiguous  escalation       True          40
                    no_escalation    True          40
         clear      escalation       True          40
                    no_escalation    True          40
```

---

## LLM run #3 (OR version)



# title
Die Entwicklung eines KI-Systems zur Eskalation klinischer Berichte erfordert aufgrund der Kritikalität (Patientensicherheit) 
eine hohe Sensitivität (Recall). Im Jahr 2026 stehen Ihnen hierfür fortschrittliche Methoden zur Verfügung, um die Genauigkeit 
und Erklärbarkeit zu maximieren.

## Strategische Ansätze zur Modellentwicklung
### Domänenspezifische Foundation Models:
Nutzen Sie keine allgemeinen Sprachmodelle, sondern spezialisierte Modelle wie Med-PaLM 3, Med-Llama oder BioBERT. 
Diese sind auf medizinischer Fachliteratur vortrainiert und verstehen medizinische Fachterminologie sowie Abkürzungen 
deutlich präziser.

### Hybrid-Ansatz (RAG + Fine-Tuning):
Fine-Tuning: Trainieren Sie das Modell mit Ihren spezifischen, annotierten Berichten, um den lokalen Stil und die 
Dringlichkeitskriterien Ihres Hauses zu lernen.
Retrieval-Augmented Generation (RAG): Verknüpfen Sie das Modell mit einer Wissensdatenbank aktueller klinischer Leitlinien. 
So kann die KI ihre Entscheidung ("Eskalation nötig") mit Verweisen auf offizielle Protokolle begründen.

### Multi-modale Datenintegration:
Eskalationsentscheidungen basieren oft nicht nur auf Text. Integrieren Sie strukturierte Labordaten oder Vitalparameter über 
Dienste wie AWS HealthLake, um den Kontext des Berichts zu validieren.

## Änderungen und Optimierungen, die Sie testen sollten
### Chain-of-Thought (CoT) Prompting: 
Zwingen Sie das Modell, den klinischen Fall Schritt für Schritt zu analysieren, bevor es eine 
Entscheidung trifft. Dies verbessert die logische Herleitung bei komplexen Diagnosen massiv.

### Human-in-the-Loop (Active Learning): 
Implementieren Sie ein System, bei dem Mediziner falsch klassifizierte Berichte korrigieren.
Diese Korrekturen fließen direkt als neue Trainingsdaten zurück in das Modell (Amazon SageMaker Ground Truth bietet hierfür Workflows).

### Anpassung der Decision Threshold: 
Da ein "übersehener" Notfall schwerwiegender ist als ein Fehlalarm, sollten Sie den Schwellenwert 
der Wahrscheinlichkeit so anpassen, dass das Modell eher zu viel als zu wenig eskaliert (Optimierung auf den F2-Score).
Interpretierbarkeit-Layer: Nutzen Sie Techniken wie SHAP oder LIME, um visualisieren zu können, welche Wörter im Bericht 
(z. B. "tachykard", "akut", "Verdacht auf...") die Eskalation ausgelöst haben.

---

```
Können wir am Anfang beginnen. ICh nutze derzeit eine Vorgabe für 'System-Prompt', 'JSON-SChema', 'allowed_values' und einen 'post-processing'-Schritt, mit dem KI-geplante Eskalationen noch deeskaliert werden können. Was hältst du von diesem Ansatz?
```

Dieser Ansatz ist architektonisch sehr solide und entspricht dem aktuellen „Best Practice“ für sicherheitskritische Anwendungen im Jahr 2026. Sie nutzen eine strukturierte Pipeline, die über reines Prompting hinausgeht.
Hier ist eine Analyse Ihrer Komponenten und wo die Fallstricke liegen:
**1. System-Prompt & JSON-Schema (Die Struktur)**   
- Vorteil: Die Kombination erzwingt deterministische Ausgaben. Ein festes Schema (z. B. via JSON Mode oder Pydantic) ist essenziell für die Integration in Klinik-Software.
- Kritik: Ein System-Prompt allein stößt bei komplexen medizinischen Grenzfällen oft an Grenzen („Prompt Brittleness“).
Empfehlung: Nutzen Sie im Prompt die „Chain-of-Thought“-Technik. Fordern Sie die KI im Schema auf, zuerst eine klinische_begruendung zu schreiben, bevor das Feld eskalation_nötig: true/false ausgefüllt wird. Das zwingt das Modell zur internen Logikprüfung.

**2. Allowed Values (Die Leitplanken)**   
- Vorteil: Verhindert Halluzinationen bei der Kategorisierung (z. B. Dringlichkeitsstufen 1–4).
Gefahr: „Label Bias“. Wenn ein Fall „zwischen“ zwei Werten liegt, wählt die KI oft den konservativeren oder den statistisch häufigeren Wert, was die Präzision senkt.

**3. Post-Processing (Die De-Eskalations-Logik)**   
- Vorteil: Dies ist Ihr wichtigster Sicherheitsfilter, um Alarm-Müdigkeit (Alarm Fatigue) bei Medizinern zu verhindern.
Das Problem: Wenn die KI eine Eskalation plant, sieht sie oft subtile sprachliche Nuancen (z. B. „Patient wirkt desorientiert“), die eine starre Regel im Post-Processing (z. B. „Nur eskalieren, wenn Puls > 100“) vielleicht wieder löscht.   
- Besserer Ansatz: Statt einer harten De-Eskalation durch Code sollten Sie ein „Critique-Loop“-Verfahren nutzen. Ein zweites, kleineres und spezialisiertes Modell (oder ein zweiter Prompt-Durchlauf) prüft die Entscheidung des ersten Modells: „Modell A will eskalieren. Prüfe, ob dies gemäß Leitlinie X wirklich nötig ist oder ob es ein Fehlalarm ist.“

Was in Ihrem aktuellen Stack noch fehlt (2026-Standard):
- Verschlagwortung (Evidence Tagging): Die KI sollte im JSON-Schema verpflichtet werden, die exakten Textstellen aus dem Bericht zu zitieren, die zur Eskalationsentscheidung geführt haben. Das schafft Vertrauen beim Arzt.   
- Kalibrierung auf „Recall“: In der Medizin ist ein False Negative (Notfall übersehen) fatal, ein False Positive (unnötige Prüfung) nur lästig. Ihr Post-Processing sollte statistisch so kalibriert sein, dass es die Sensitivität (Recall) nicht unter 99% drückt.   
- Vektordatenbank-Abgleich (RAG): Vergleichen Sie den aktuellen Bericht mit „Gold Standard“-Berichten aus Ihrer Datenbank, bevor die Entscheidung finalisiert wird.
